# Sketch2Spec backend on Google Colab

Backend นี้เปิดใช้งานผ่านลิงก์โดยไม่มี login และไม่แก้ `backend/best.pt`. ก่อนรันให้เปลี่ยน `VERCEL_ORIGIN` เป็น URL Vercel จริง.

In [ ]:
!pkill -f uvicorn || true
!pkill -f cloudflared || true
!rm -rf Sketch2Spec
!git clone --depth 1 https://github.com/kankanitsen/Sketch2Spec.git
%cd /content/Sketch2Spec
!pip install -q -r backend/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('ติดตั้งเสร็จแล้ว')

In [ ]:
import os

VERCEL_ORIGIN = 'https://sketch2-spec.vercel.app'  # เปลี่ยนเป็น URL Vercel ของคุณ
if 'YOUR_PROJECT' in VERCEL_ORIGIN:
    raise ValueError('กรุณาใส่ URL Vercel จริงใน VERCEL_ORIGIN')
os.environ['ALLOWED_ORIGINS'] = VERCEL_ORIGIN.rstrip('/')
print('อนุญาต frontend:', os.environ['ALLOWED_ORIGINS'])

In [ ]:
import os, re, time, subprocess

os.makedirs('/content/logs', exist_ok=True)
api_log = open('/content/logs/uvicorn.log', 'w')
tunnel_log = open('/content/logs/cloudflared.log', 'w')
subprocess.Popen(['uvicorn', 'main:app', '--app-dir', 'backend', '--host', '0.0.0.0', '--port', '8000'], stdout=api_log, stderr=subprocess.STDOUT)
subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(30):
    time.sleep(2)
    log = open('/content/logs/cloudflared.log').read() if os.path.exists('/content/logs/cloudflared.log') else ''
    found = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', log)
    if found:
        public_url = found.group(0)
        break
if not public_url:
    raise RuntimeError('สร้าง tunnel ไม่สำเร็จ: ดู /content/logs/cloudflared.log')
print('Backend URL:', public_url)
print('นำ URL นี้ไปใส่ใน Vercel > NEXT_PUBLIC_DETECTION_API_URL แล้ว Redeploy')

In [ ]:
import time, requests
for _ in range(20):
    try:
        response = requests.get('http://127.0.0.1:8000/health', timeout=3)
        if response.ok:
            print('Health ตรวจผ่าน:', response.json())
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError(open('/content/logs/uvicorn.log').read())